In [ ]:
# Try implement CLIP (source: https://arxiv.org/pdf/2103.00020)

from transformers import ViTImageProcessor, ViTModel
from sentence_transformers import SentenceTransformer
from PIL import Image
import requests
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'


c:\Users\taput\anaconda3\envs\chatbot-fraudster\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
vision_model = ViTModel.from_pretrained('google/vit-base-patch16-224').to(device)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1924.12it/s]
[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
text_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B").to(device)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1084.61it/s]


In [4]:
inputs = processor(images=image, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = vision_model(**inputs)

In [5]:
image_embeddings = outputs.last_hidden_state[:,0]
# image_embeddings = torch.cat([image_embedding, image_embedding, image_embedding], dim=0)

In [6]:
documents = [
    "This is a cat",
    "This is a dog",
    "This is a bird",
]
document_embeddings = text_model.encode(documents, convert_to_tensor=True).clone().float()

In [ ]:
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
class CLIP(nn.Module):
    def __init__(self, img_feature_size: int, txt_feature_size: int, output_size: int, temperature: float=0.2):
        super().__init__()
        self.image_embed_layer = nn.Linear(img_feature_size, output_size, bias=False)
        self.txt_embed_layer = nn.Linear(txt_feature_size, output_size, bias=False)
        self.temperature = nn.Parameter(torch.tensor(np.log(1/temperature)))

    def forward(self, img_feature, txt_feature):
        img_embed = self.image_embed_layer(img_feature)
        txt_embed = self.txt_embed_layer(txt_feature)

        # Euclidean distance is p=2
        img_embed = F.normalize(img_embed, p=2, dim=1)
        txt_embed = F.normalize(txt_embed, p=2, dim=1)

        logits = img_embed @ txt_embed.transpose(0, 1) * torch.exp(self.temperature)
        return logits

In [20]:
clip = CLIP(image_embeddings.shape[1], document_embeddings.shape[1], 64).to(device)

In [21]:
logits = clip(image_embeddings, document_embeddings)

In [22]:
label = torch.tensor([0]).to(device)
loss = F.cross_entropy(logits, label)

In [23]:
loss

tensor(1.1405, device='cuda:0', grad_fn=<NllLossBackward0>)